In [ ]:
import pandas as pd  #basic, import pandas library


#Display settings 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Load data
df = pd.read_csv('nyc_2026_1st_quarter_data.csv')


#calculating room count ,later used in filling missing data / cleaning
df["room_count"] = df[["beds", "baths_full_calc", "garage"]].sum(axis=1)

#aggregation for later use
total_rooms = df['room_count'].sum()
total_price = df['listPrice'].sum()
total_sqft = df['sqft'].sum()
total_stories = df['stories'].sum()

avgpriceperroom = total_price / total_rooms
avgsqftperroom = total_sqft / total_rooms
avgroomsperstory = total_rooms / total_stories

# 3️⃣ Fill missing columns
# Fill missing sqft based on room count
fill_sqft = df['sqft'].isna() & df['room_count'].notna()
df.loc[fill_sqft, 'sqft'] = df.loc[fill_sqft, 'room_count'] * avgsqftperroom

# Fill missing stories based on room count
fill_stories = df['stories'].isna() & df['room_count'].notna()
df.loc[fill_stories, 'stories'] = df.loc[fill_stories, 'room_count'] / avgroomsperstory

# Fill missing garage with 1
df.loc[df['garage'].isna(), 'garage'] = 1

# -------------------------
# 4️⃣ Calculate price per sqft
# -------------------------
# Initial calculation
df['price_per_sqft'] = df['listPrice'] / df['sqft']

# Fill any remaining NaNs with market average
market_avg_sqft = df['listPrice'].sum() / df['sqft'].sum()
df.loc[df['price_per_sqft'].isna(), 'price_per_sqft'] = market_avg_sqft

# -------------------------
# 5️⃣ Compare each property to market
# -------------------------
df.loc[df['price_per_sqft'] > market_avg_sqft, 'market_value'] = 'Above Market Value'
df.loc[df['price_per_sqft'] < market_avg_sqft, 'market_value'] = 'Below Market Value'
df.loc[df['price_per_sqft'] == market_avg_sqft, 'market_value'] = 'At Market Value'




unrealistic = (
    (df['listPrice'] < 75000) | (df['listPrice'] > 10000000) & (df['sqft'] < 10000) |
    (df['sqft'] < 300) | (df['sqft'] > 20000) |
    (df['beds'] < 1) | (df['beds'] > 20) |
    (df['baths_full_calc'] < 1) | (df['stories'] < 1) |
    (df['garage'] < 0) | (df['garage'] > 5) |
    (df['price_per_sqft'] <= 0) | (df['price_per_sqft'] > 3000) | (df['room_count'] < 2) | (df['room_count'] > 30) |
    (~df['price_per_sqft'].replace([float('inf'), float('-inf')], float('nan')).notna())
)

df_cleaned = df[~unrealistic].copy()

df_cleaned.to_csv('NYC_dataset_cleaned_Amr_Eltaieby.csv', index=False)
